# Cohort Construction

Rebuilds the analytic cohort directly from the All of Us CDR, applies the manuscript follow-up eligibility rule, and saves the participant-level cohort used by every downstream analysis.

This notebook stops after saving `paper_cohort.parquet` in the configured data directory (default `~/lungcancer`). PheCode construction and all predictive modeling are performed in Notebook 03.

## Important endpoint note

The manuscript flowchart defines a **3-year lung cancer** endpoint and excludes controls with less than 3 years of follow-up. The paper definition is treated as authoritative.

Do not commit generated participant-level files to Git. They remain under the configured data directory inside the controlled Workbench environment.


In [ ]:
import os
import re
import shutil
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

warnings.filterwarnings("ignore")

datadir = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
datadir.mkdir(parents=True, exist_ok=True)

PAPER_HORIZON_YEARS = 3
PAPER_DATA_CUTOFF = pd.Timestamp("2023-10-01")
CDR_RESOURCE_ID = os.environ.get("LUNG_CANCER_CDR_RESOURCE_ID", "").strip()


def resolve_workspace_cdr():
    legacy_cdr = os.environ.get("WORKSPACE_CDR", "").strip()
    if legacy_cdr:
        return legacy_cdr.removeprefix("bq://").replace(":", ".", 1)

    if not shutil.which("wb"):
        raise RuntimeError(
            "No WORKSPACE_CDR environment variable or Workbench 2 `wb` CLI was found. "
            "Run this notebook inside the All of Us Researcher Workbench."
        )
    if not CDR_RESOURCE_ID:
        listing = subprocess.run(
            ["wb", "resource", "list"],
            check=True, capture_output=True, text=True,
        ).stdout
        raise RuntimeError(
            "Workbench 2 does not inject WORKSPACE_CDR. Run `wb resource list`, "
            "then set LUNG_CANCER_CDR_RESOURCE_ID to the CDR dataset resource ID "
            "and restart this cell.\n\nCurrent workspace resources:\n" + listing
        )

    output = subprocess.run(
        [
            "wb", "resource", "resolve",
            f"--id={CDR_RESOURCE_ID}",
            "--bq-path=FULL_PATH",
            "--format=TEXT",
        ],
        check=True, capture_output=True, text=True,
    ).stdout
    lines_out = [line.strip() for line in output.splitlines() if line.strip()]
    if not lines_out:
        raise RuntimeError(
            f"Workbench returned no BigQuery path for resource {CDR_RESOURCE_ID!r}."
        )
    return lines_out[-1].strip("`'\"").removeprefix("bq://").replace(":", ".", 1)


WORKSPACE_CDR = resolve_workspace_cdr()

print("CDR:", WORKSPACE_CDR)
print("Data directory:", datadir)
print("Paper-reproduction horizon:", PAPER_HORIZON_YEARS, "years")
print("Paper administrative data cutoff:", PAPER_DATA_CUTOFF.date())


## 1. Shared helpers

In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def to_datetime_naive(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None)


def norm_text(s):
    if pd.isna(s):
        return ""
    return re.sub(r"\s+", " ", str(s).strip().lower())


def add_race_ethnicity(df):
    if "race_ethnicity" in df.columns:
        df["race_ethnicity"] = df["race_ethnicity"].fillna("Missing")
        return df
    if not {"race", "ethnicity"}.issubset(df.columns):
        df["race_ethnicity"] = "Missing"
        return df

    race_missing = {"none indicated", "i prefer not to answer", "pmi: skip", "none of these", ""}
    eth_missing = {
        "no matching concept",
        "pmi: prefer not to answer",
        "pmi: skip",
        "what race ethnicity: race ethnicity none of these",
        "",
    }

    def make(row):
        race = norm_text(row.get("race"))
        eth = norm_text(row.get("ethnicity"))
        if eth == "hispanic or latino":
            return "Hispanic"
        if eth in eth_missing:
            return "Missing"
        if eth == "not hispanic or latino":
            if race == "white":
                return "White"
            if race == "black or african american":
                return "Black"
            if race == "asian":
                return "Asian"
            if race == "american indian or alaska native":
                return "American Indian or Alaskan Native"
            if race == "native hawaiian or other pacific islander":
                return "Native Hawaiian or Pacific Islander"
            if race in race_missing:
                return "Missing"
        return "Missing"

    df["race_ethnicity"] = df.apply(make, axis=1)
    return df


def add_age_at_survey(df):
    if "age_at_survey_int" in df.columns:
        return df
    if {"date_of_birth", "lifestyle_survey_date"}.issubset(df.columns):
        dob = to_datetime_naive(df["date_of_birth"])
        survey = to_datetime_naive(df["lifestyle_survey_date"])
        df["age_at_survey"] = (survey - dob).dt.days / 365.25
        df["age_at_survey_int"] = df["age_at_survey"].round(0).astype("Int64")
    return df


def add_3y_outcome(df):
    if "has_lungcancer_3Y" in df.columns:
        df["has_lungcancer_3Y"] = df["has_lungcancer_3Y"].astype(int)
        return df
    required = {"lungcancer_date", "lifestyle_survey_date"}
    if not required.issubset(df.columns):
        raise ValueError("Need has_lungcancer_3Y or lungcancer_date + lifestyle_survey_date.")
    survey = to_datetime_naive(df["lifestyle_survey_date"])
    lung = to_datetime_naive(df["lungcancer_date"])
    df["has_lungcancer_3Y"] = (
        lung.notna()
        & (lung > survey)
        & (lung <= survey + pd.DateOffset(years=PAPER_HORIZON_YEARS))
    ).astype(int)
    return df


def remove_prior_lung_cancer(df):
    if not {"lungcancer_date", "lifestyle_survey_date"}.issubset(df.columns):
        return df
    survey = to_datetime_naive(df["lifestyle_survey_date"])
    lung = to_datetime_naive(df["lungcancer_date"])
    prior = lung.notna() & (lung < survey)
    print("Removing prior lung cancer rows:", int(prior.sum()))
    return df.loc[~prior].copy()



## 2. Extract demographics, outcomes, and lifestyle variables

In [ ]:
def read_gbq(sql):
    return pd.read_gbq(
        sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )


def require_all_of_us():
    return WORKSPACE_CDR


def first_per_person(df, date_col):
    return (
        df.sort_values(["person_id", date_col])
        .drop_duplicates("person_id")
        .reset_index(drop=True)
    )


def ever_smoker_filter_sql(workspace_cdr):
    return f"""
    SELECT DISTINCT person_id
    FROM `{workspace_cdr}.cb_search_all_events`
    WHERE concept_id IN (1585857)
      AND is_standard = 0
      AND value_source_concept_id IN (1585858)
    """


def build_lifestyle_tables(workspace_cdr):
    print("Querying Lifestyle survey answers...")
    lifestyle_sql = f"""
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.question,
        answer.answer
    FROM `{workspace_cdr}.ds_survey` answer
    WHERE answer.question_concept_id IN (
        SELECT DISTINCT concept_id
        FROM `{workspace_cdr}.cb_criteria` c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM `{workspace_cdr}.cb_criteria` cr
            WHERE concept_id IN (1585855)
              AND domain_id = 'SURVEY'
        ) a
          ON c.path LIKE CONCAT('%', a.id, '.%')
    )
      AND answer.person_id IN ({ever_smoker_filter_sql(workspace_cdr)})
    """
    lifestyle = read_gbq(lifestyle_sql)
    lifestyle["survey_datetime"] = pd.to_datetime(lifestyle["survey_datetime"], errors="coerce", utc=True).dt.tz_convert(None)
    lifestyle = lifestyle.dropna(subset=["person_id", "survey_datetime"]).copy()

    first_lifestyle = (
        lifestyle.sort_values("survey_datetime")
        .groupby("person_id", as_index=False)
        .first()[["person_id", "survey_datetime"]]
        .rename(columns={"survey_datetime": "lifestyle_survey_date"})
    )

    lifestyle_wide = (
        lifestyle.sort_values("survey_datetime")
        .drop_duplicates(["person_id", "question"])
        .pivot(index="person_id", columns="question", values="answer")
        .reset_index()
    )

    lifestyle.to_parquet(datadir / "lifestyle.parquet", index=False, compression="snappy")
    lifestyle_wide.to_parquet(datadir / "lifestyle_wide.parquet", index=False, compression="snappy")
    return lifestyle, lifestyle_wide, first_lifestyle


def lifestyle_person_filter_sql(workspace_cdr):
    return f"""
    SELECT DISTINCT answer.person_id
    FROM `{workspace_cdr}.ds_survey` answer
    WHERE answer.question_concept_id IN (
        SELECT DISTINCT concept_id
        FROM `{workspace_cdr}.cb_criteria` c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM `{workspace_cdr}.cb_criteria` cr
            WHERE concept_id IN (1585855)
              AND domain_id = 'SURVEY'
        ) a
          ON c.path LIKE CONCAT('%', a.id, '.%')
    )
      AND answer.person_id IN ({ever_smoker_filter_sql(workspace_cdr)})
    """


def add_lifestyle_covariates(demo, lifestyle_wide):
    demo = demo.copy()
    lw = lifestyle_wide.copy()

    edu_map = {
        "Highest Grade: Never Attended": "Less than high school",
        "Highest Grade: One Through Four": "Less than high school",
        "Highest Grade: Five Through Eight": "Less than high school",
        "Highest Grade: Nine Through Eleven": "Less than high school",
        "Highest Grade: Twelve Or GED": "High school or GED",
        "Highest Grade: College One to Three": "Some college",
        "Highest Grade: College Graduate": "College graduate",
        "Highest Grade: Advanced Degree": "Advanced degree",
        "PMI: Skip": "Missing",
        "PMI: Prefer Not To Answer": "Missing",
    }
    if "Education: Highest Grade" in lw.columns:
        lw["education_group"] = lw["Education: Highest Grade"].map(edu_map).fillna("Missing")
    elif "Highest Grade" in lw.columns:
        lw["education_group"] = lw["Highest Grade"].map(edu_map).fillna("Missing")
    else:
        lw["education_group"] = "Missing"

    for col in [
        "Attempt Quit Smoking: Completely Quit Age",
        "Smoking: Daily Smoke Starting Age",
        "Smoking: Number Of Years",
        "Smoking: Current Daily Cigarette Number",
        "Smoking: Average Daily Cigarette Number",
    ]:
        if col in lw.columns:
            lw[col] = pd.to_numeric(lw[col], errors="coerce")

    if {"Attempt Quit Smoking: Completely Quit Age", "Smoking: Daily Smoke Starting Age"}.issubset(lw.columns):
        lw["duration_of_quitting"] = (
            lw["Attempt Quit Smoking: Completely Quit Age"]
            - lw["Smoking: Daily Smoke Starting Age"]
        )
        lw.loc[lw["duration_of_quitting"] < 0, "duration_of_quitting"] = np.nan
    else:
        lw["duration_of_quitting"] = np.nan

    if "Smoking: Smoke Frequency" in lw.columns:
        # CDR releases vary in capitalization and may use the full concept label,
        # e.g. "Smoking: Smoke Frequency Every Day" instead of "Every Day".
        freq = (
            lw["Smoking: Smoke Frequency"]
            .astype("string")
            .str.strip()
            .str.lower()
        )
        former_mask = freq.str.contains(r"not\s+at\s+all", regex=True, na=False)
        current_mask = freq.str.contains(r"every\s+day|some\s+days", regex=True, na=False)
        lw["smoking_status"] = np.select(
            [former_mask.to_numpy(dtype=bool), current_mask.to_numpy(dtype=bool)],
            ["former", "current"],
            default="missing",
        )
    else:
        lw["smoking_status"] = "missing"

    covars = [
        "person_id",
        "education_group",
        "duration_of_quitting",
        "Attempt Quit Smoking: Completely Quit Age",
        "Smoking: Number Of Years",
        "Smoking: Current Daily Cigarette Number",
        "Smoking: Average Daily Cigarette Number",
        "smoking_status",
    ]
    for col in covars:
        if col not in lw.columns:
            lw[col] = np.nan

    print("Smoking status after parsing:")
    print(lw["smoking_status"].value_counts(dropna=False))
    print("Nonmissing lifetime cigarettes/day:", int(lw["Smoking: Average Daily Cigarette Number"].notna().sum()))
    print("Nonmissing current cigarettes/day:", int(lw["Smoking: Current Daily Cigarette Number"].notna().sum()))
    print("Nonmissing smoking years:", int(lw["Smoking: Number Of Years"].notna().sum()))
    print("Nonmissing quit age:", int(lw["Attempt Quit Smoking: Completely Quit Age"].notna().sum()))

    if not lw["smoking_status"].isin(["current", "former"]).any():
        raw_examples = (
            lifestyle_wide.get("Smoking: Smoke Frequency", pd.Series(dtype="string"))
            .dropna().astype(str).value_counts().head(10).to_dict()
        )
        raise RuntimeError(
            "Could not recognize any current/former smoking-status labels. "
            f"Observed labels: {raw_examples}"
        )
    return demo.merge(lw[covars], on="person_id", how="left")


def build_base_cohort_from_bigquery():
    workspace_cdr = require_all_of_us()

    _, lifestyle_wide, first_lifestyle = build_lifestyle_tables(workspace_cdr)
    cohort_filter = lifestyle_person_filter_sql(workspace_cdr)

    print("Querying demographics...")
    person_sql = f"""
    SELECT
        person.person_id,
        person.birth_datetime AS date_of_birth,
        p_race_concept.concept_name AS race,
        p_ethnicity_concept.concept_name AS ethnicity,
        p_sex_at_birth_concept.concept_name AS sex_at_birth
    FROM `{workspace_cdr}.person` person
    LEFT JOIN `{workspace_cdr}.concept` p_race_concept
      ON person.race_concept_id = p_race_concept.concept_id
    LEFT JOIN `{workspace_cdr}.concept` p_ethnicity_concept
      ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
    LEFT JOIN `{workspace_cdr}.concept` p_sex_at_birth_concept
      ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    WHERE person.person_id IN (
      SELECT person_id FROM `{workspace_cdr}.cb_search_person` WHERE has_ehr_data = 1
    )
    AND person.person_id IN ({cohort_filter})
    """
    demo = read_gbq(person_sql)
    demo = demo.merge(first_lifestyle, on="person_id", how="inner")
    demo["lifestyle_survey_date"] = to_datetime_naive(demo["lifestyle_survey_date"])
    demo = add_age_at_survey(add_race_ethnicity(demo))

    print("Querying lung cancer dates...")
    lung_sql = f"""
    SELECT
      co.person_id,
      MIN(co.condition_start_datetime) AS lungcancer_date
    FROM `{workspace_cdr}.condition_occurrence` co
    WHERE co.person_id IN ({cohort_filter})
      AND (
        co.condition_concept_id IN (
          SELECT descendant_concept_id
          FROM `{workspace_cdr}.concept_ancestor`
          WHERE ancestor_concept_id = 443388
        )
        OR co.condition_concept_id = 443388
        OR co.condition_source_concept_id IN (1567486, 1567490, 1567491, 35206184, 44820606, 44825200, 44826401, 44827554, 44827555, 44829802, 44830972, 44835673, 45532899, 45542591, 45547486, 45556973, 45561794, 45566582, 45571493, 45571494, 45576310, 45581204, 45581205, 45581206, 45590884, 45590885, 45600488)
      )
    GROUP BY co.person_id
    """
    lung = read_gbq(lung_sql)
    demo = demo.merge(lung, on="person_id", how="left")
    demo["lungcancer_date"] = to_datetime_naive(demo["lungcancer_date"])
    demo = add_3y_outcome(demo)
    demo["has_lungcancer"] = demo["lungcancer_date"].notna().astype(int)
    demo["has_lungcancer_post"] = (
        demo["lungcancer_date"].notna()
        & (demo["lungcancer_date"] >= demo["lifestyle_survey_date"])
    ).astype(int)
    demo["has_lungcancer_12M"] = (
        demo["lungcancer_date"].notna()
        & (demo["lungcancer_date"] >= demo["lifestyle_survey_date"])
        & (demo["lungcancer_date"] <= demo["lifestyle_survey_date"] + pd.DateOffset(months=12))
    ).astype(int)
    demo["has_lungcancer_24M"] = (
        demo["lungcancer_date"].notna()
        & (demo["lungcancer_date"] >= demo["lifestyle_survey_date"])
        & (demo["lungcancer_date"] <= demo["lifestyle_survey_date"] + pd.DateOffset(months=24))
    ).astype(int)
    demo["has_lungcancer_6Y"] = (
        demo["lungcancer_date"].notna()
        & (demo["lungcancer_date"] >= demo["lifestyle_survey_date"])
        & (demo["lungcancer_date"] <= demo["lifestyle_survey_date"] + pd.DateOffset(years=6))
    ).astype(int)
    demo = remove_prior_lung_cancer(demo)

    print("Querying COPD dates...")
    copd_sql = f"""
    SELECT
      co.person_id,
      MIN(co.condition_start_datetime) AS copd_date
    FROM `{workspace_cdr}.condition_occurrence` co
    WHERE co.person_id IN ({cohort_filter})
      AND (
        co.condition_concept_id IN (
          SELECT descendant_concept_id
          FROM `{workspace_cdr}.concept_ancestor`
          WHERE ancestor_concept_id IN (255573, 4110056)
        )
        OR co.condition_concept_id IN (255573, 4110056)
      )
    GROUP BY co.person_id
    """
    copd = read_gbq(copd_sql)
    demo = demo.merge(copd, on="person_id", how="left")
    demo["copd_date"] = to_datetime_naive(demo["copd_date"])
    demo["COPD_pre"] = (
        demo["copd_date"].notna()
        & (demo["copd_date"] < demo["lifestyle_survey_date"])
    ).astype(int)

    print("Querying BMI measurements...")
    bmi_sql = f"""
    SELECT person_id, measurement_datetime, value_as_number AS bmi
    FROM `{workspace_cdr}.measurement`
    WHERE person_id IN ({cohort_filter})
      AND measurement_concept_id = 3038553
      AND value_as_number IS NOT NULL
    """
    bmi = read_gbq(bmi_sql)
    if len(bmi) > 0:
        bmi["measurement_datetime"] = to_datetime_naive(bmi["measurement_datetime"])
        bmi = bmi.merge(demo[["person_id", "lifestyle_survey_date"]], on="person_id", how="inner")
        bmi["abs_days"] = (bmi["measurement_datetime"] - bmi["lifestyle_survey_date"]).abs().dt.days
        bmi = first_per_person(bmi.sort_values("abs_days"), "abs_days")[["person_id", "bmi"]]
        demo = demo.merge(bmi, on="person_id", how="left")
    else:
        demo["bmi"] = np.nan

    print("Querying family history of lung cancer...")
    family_sql = f"""
    SELECT DISTINCT person_id
    FROM `{workspace_cdr}.observation`
    WHERE person_id IN ({cohort_filter})
      AND observation_source_concept_id = 35225306
    """
    family_lung = read_gbq(family_sql)
    demo["family_lung"] = demo["person_id"].isin(family_lung["person_id"]).astype(int)

    demo = add_lifestyle_covariates(demo, lifestyle_wide)
    demo["cancer_history"] = 0

    out = datadir / "demo_3y.parquet"
    demo.to_parquet(out, index=False, compression="snappy")
    print("Saved rebuilt base cohort:", out, demo.shape)
    print("Cases:", int(demo["has_lungcancer_3Y"].sum()))
    return demo



In [ ]:
# Always rebuild so the flow counts correspond to the active CDR release.
demo = build_base_cohort_from_bigquery()

print("After lifestyle + EHR + ever-smoker criteria and prior-cancer exclusion:")
print("N:", len(demo))
print("Paper-horizon cases before follow-up filtering:", int(demo["has_lungcancer_3Y"].sum()))


## 3. Reconstruct EHR observation bounds

In [ ]:
def query_ehr_observation_bounds(person_ids, chunk_size=5000):
    # Compute first/last recorded EHR activity across core OMOP clinical domains.
    client = bigquery.Client()
    cdr = WORKSPACE_CDR
    ids = pd.Series(person_ids).dropna().astype("int64").drop_duplicates().tolist()
    pieces = []

    for start in range(0, len(ids), chunk_size):
        chunk = ids[start:start + chunk_size]
        id_sql = ",".join(map(str, chunk))
        sql = f"""
        WITH ehr_events AS (
          SELECT person_id, condition_start_datetime AS event_datetime
          FROM `{cdr}.condition_occurrence`
          WHERE person_id IN ({id_sql})
          UNION ALL
          SELECT person_id, drug_exposure_start_datetime
          FROM `{cdr}.drug_exposure`
          WHERE person_id IN ({id_sql})
          UNION ALL
          SELECT person_id, visit_start_datetime
          FROM `{cdr}.visit_occurrence`
          WHERE person_id IN ({id_sql})
          UNION ALL
          SELECT person_id, procedure_datetime
          FROM `{cdr}.procedure_occurrence`
          WHERE person_id IN ({id_sql})
          UNION ALL
          SELECT person_id, measurement_datetime
          FROM `{cdr}.measurement`
          WHERE person_id IN ({id_sql})
          UNION ALL
          SELECT person_id, observation_datetime
          FROM `{cdr}.observation`
          WHERE person_id IN ({id_sql})
          UNION ALL
          SELECT person_id, device_exposure_start_datetime
          FROM `{cdr}.device_exposure`
          WHERE person_id IN ({id_sql})
        )
        SELECT
          person_id,
          MIN(event_datetime) AS first_ehr_date,
          MAX(event_datetime) AS last_ehr_date
        FROM ehr_events
        WHERE event_datetime IS NOT NULL
        GROUP BY person_id
        """
        piece = client.query(sql).result().to_dataframe(create_bqstorage_client=True)
        pieces.append(piece)
        print(f"EHR date chunk {start // chunk_size + 1}: {min(start + chunk_size, len(ids)):,}/{len(ids):,}")

    return pd.concat(pieces, ignore_index=True)


ehr_bounds = query_ehr_observation_bounds(demo["person_id"])
demo = demo.drop(columns=["first_ehr_date", "last_ehr_date"], errors="ignore").merge(
    ehr_bounds,
    on="person_id",
    how="left",
)

for col in ["lifestyle_survey_date", "lungcancer_date", "first_ehr_date", "last_ehr_date"]:
    demo[col] = to_datetime_naive(demo[col])

print("Participants with EHR observation bounds:", int(demo["last_ehr_date"].notna().sum()))


## 4. Apply the manuscript flowchart eligibility rule

In [ ]:
survey = demo["lifestyle_survey_date"]
lung = demo["lungcancer_date"]

paper_endpoint = survey + pd.DateOffset(years=3)
# Cap current-CDR activity at the paper-era administrative endpoint.
demo["effective_last_ehr_date"] = demo["last_ehr_date"].clip(upper=PAPER_DATA_CUTOFF)
legacy_4y_endpoint = survey + pd.DateOffset(years=4)

demo["has_lungcancer_3Y"] = (
    lung.notna()
    & (lung > survey)
    & (lung <= paper_endpoint)
).astype(int)

# Diagnostic only: this reproduces the horizon in the mislabeled exploratory cell.
demo["has_lungcancer_legacy_4Y"] = (
    lung.notna()
    & (lung > survey)
    & (lung <= legacy_4y_endpoint)
).astype(int)

# Manuscript rule: retain cases within 3 years; controls must remain observable
# for at least 3 years after the lifestyle survey.
eligible = (
    (demo["has_lungcancer_3Y"] == 1)
    | (
        (demo["has_lungcancer_3Y"] == 0)
        & (demo["effective_last_ehr_date"] >= paper_endpoint)
    )
)

paper_cohort = demo.loc[eligible].copy().reset_index(drop=True)

flow_check = pd.DataFrame([
    {"stage": "After prior lung cancer exclusion", "observed_n": len(demo), "paper_n": 152550},
    {"stage": "Excluded for insufficient 3-year follow-up", "observed_n": int((~eligible).sum()), "paper_n": 61726},
    {"stage": "Final analytic cohort", "observed_n": len(paper_cohort), "paper_n": 90824},
    {"stage": "Lung cancer within 3 years", "observed_n": int(paper_cohort["has_lungcancer_3Y"].sum()), "paper_n": 953},
    {"stage": "No lung cancer within 3 years", "observed_n": int((paper_cohort["has_lungcancer_3Y"] == 0).sum()), "paper_n": 89871},
])
flow_check["difference"] = flow_check["observed_n"] - flow_check["paper_n"]

print("Raw last EHR maximum:", demo["last_ehr_date"].max())
print("Administrative cutoff:", PAPER_DATA_CUTOFF)
print("True 3-year cases before follow-up filtering:", int(demo["has_lungcancer_3Y"].sum()))
print("Legacy four-year diagnostic cases:", int(demo["has_lungcancer_legacy_4Y"].sum()))
display(flow_check)

demo = paper_cohort
demo.to_parquet(datadir / "paper_cohort.parquet", index=False, compression="snappy")


### Interpreting discrepancies

Exact equality requires the same CDR release, cohort concept set, index-date rule, and EHR activity definition used for the paper. Do not force the current cohort to the published sample size by randomly dropping participants. Review the printed differences instead.


## Output

The reconstructed cohort is saved as `paper_cohort.parquet` in the configured data directory. Run Notebook 02 for clinical benchmarks and Notebook 03 for PheCode feature construction and modeling.
